# ROGII Wellbore Geology - Dense (MLP) Training

Train a Multi-Layer Perceptron (Dense) model to predict **TVT** from horizontal well logs.

**Features (12):** `MD, X, Y, Z, ANCC, ASTNU, ASTNL, EGFDU, EGFDL, BUDA, GR, TVT_input`

**Label:** `TVT`

**Runtime**: Kaggle GPU (T4/P100) - Keras 3 + JAX backend

**Author**: Samir Attrah

### 1. Environment & Imports
Setup JAX backend for Keras and import necessary libraries.

In [1]:
# Cell 1: Environment & Imports
import os
os.environ["KERAS_BACKEND"] = "jax"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"

import jax
# Enable JAX float64 precision natively
jax.config.update("jax_enable_x64", True)

import keras
from keras import layers, callbacks, regularizers
import jax.numpy as jnp
import numpy as np
import polars as pl
import glob, pickle, warnings, random
warnings.filterwarnings("ignore")

# Show enough digits to round-trip float64 diagnostics
np.set_printoptions(precision=17, floatmode="unique")

# Set seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
keras.utils.set_random_seed(SEED)

print(f"Keras version : {keras.__version__}")
print(f"Keras backend : {keras.backend.backend()}")
print(f"Working dir   : {os.getcwd()}")


2026-07-25 20:47:51.077665: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1785001671.093002  464443 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1785001671.098607  464443 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Keras version : 3.12.0
Keras backend : jax
Working dir   : /home/samer/Documents/competitions/ROGII/notebooks


### 2. Auto-detect Dataset Path
Locate the competition dataset in local or Kaggle environments.

In [2]:
# Cell 2: Auto-detect dataset path

def find_data_dir():
    """Searches common Kaggle mount points for the ROGII dataset."""
    candidates = [
        "/kaggle/input/competitions/rogii-wellbore-geology-prediction",
        "/kaggle/input/rogii-wellbore-geology-prediction",
        "/home/samer/Documents/competitions/ROGII/dataset",
    ]

    for scan_root in ["/kaggle/input", "/kaggle/input/competitions"]:
        if os.path.isdir(scan_root):
            for entry in os.listdir(scan_root):
                full = os.path.join(scan_root, entry)
                if os.path.isdir(full) and full not in candidates:
                    candidates.append(full)

    print("Searching for ROGII dataset...")
    for path in candidates:
        if not os.path.isdir(path):
            continue

        contents = os.listdir(path)
        has_train = "train" in contents and os.path.isdir(os.path.join(path, "train"))
        n_train = 0
        if has_train:
            n_train = len(glob.glob(os.path.join(path, "train", "*__horizontal_well.csv")))

        if n_train > 0:
            print(f"  V Using {path} (found {n_train} train wells)")
            return path

    raise FileNotFoundError("Could not find ROGII dataset.")

DATA_DIR = find_data_dir()


Searching for ROGII dataset...
  V Using /home/samer/Documents/competitions/ROGII/dataset (found 773 train wells)


### 3. Configuration
Define hyperparameters, model paths, and feature columns.

In [3]:
# Cell 3: Configuration

OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle") else os.path.abspath(os.path.join(os.getcwd(), "..", "outputs"))
os.makedirs(OUT_DIR, exist_ok=True)

CONFIG = {
    "seed": 42,
    "data_dir": DATA_DIR,
    "model_path": f"{OUT_DIR}/dense_tvt_model.keras",
    "scaler_path": f"{OUT_DIR}/dense_scaler_params.pkl",
    "hidden_layers": [128, 64],
    "dropout": 0.30,
    "kr_rate": 1e-5,
    "epochs": 100,
    "batch_size": 128,
    "lr": 5e-7,
    "val_ratio": 0.20,
    "max_wells": None,
    "gcn": 1.0,
    "beta_1": 0.9,
    "beta_2": 0.9,
    "amsgrad": False,
    "weight_decay": 0,
    "ema_momentum": 0.9,
}

FEATURE_COLS = ["MD", "X", "Y", "Z", "GR", "TVT_input"]
TARGET = "TVT"

print(f"Data dir   : {CONFIG['data_dir']}")
print(f"Model path : {CONFIG['model_path']}")
print(f"Features   : {FEATURE_COLS}")


Data dir   : /home/samer/Documents/competitions/ROGII/dataset
Model path : /home/samer/Documents/competitions/ROGII/outputs/dense_tvt_model.keras
Features   : ['MD', 'X', 'Y', 'Z', 'GR', 'TVT_input']


### 4. Data Helpers & Preparation
Functions for loading, preprocessing, and preparing the flat dataset for Dense training.

In [4]:
# Cell 4: Data helpers

def load_well(data_dir, filename, split="train", augmented=False):
    """Loads one horizontal well CSV as Polars DataFrame."""
    dataset_root = os.path.dirname(data_dir)
    # if augmented:
    #     # Load from augmented directory
    #     path = os.path.join(dataset_root, "dataset_augmented", "train", filename)
    # else:
        # Load from original directory
    path = os.path.join(data_dir, split, filename)
    return pl.read_csv(path, infer_schema_length=10000)

def preprocess(df):
    """Preprocess: Interpolation and filling nulls for all features."""
    for col in FEATURE_COLS:
        if col in df.columns:
            df = df.with_columns(
                pl.col(col).interpolate()
                  .fill_null(strategy="forward").fill_null(strategy="backward")
                  .fill_null(0.0)
            )
    return df

def prepare_data(cfg):
    """Load all wells (original + augmented), concatenate into one large flat dataset."""
    dataset_root = os.path.dirname(cfg["data_dir"])
    
    # 1. Get original training files
    pattern_train = os.path.join(cfg["data_dir"], "train", "*__horizontal_well.csv")
    files_train = sorted([os.path.basename(f) for f in glob.glob(pattern_train)])
    
    # 2. Get augmented training files (ONLY the _aug.csv ones)
    # pattern_aug = os.path.join(dataset_root, "dataset_augmented", "train", "*__horizontal_well_aug.csv")
    # files_aug = sorted([os.path.basename(f) for f in glob.glob(pattern_aug)])
    
    # Combined training file list
    files_train_all = files_train #+ files_aug
    
    # Shuffle the wells to mix original and augmented data
    np.random.seed(cfg["seed"])
    np.random.shuffle(files_train_all)
    
    if cfg["max_wells"]:
        files_train_all = files_train_all[:cfg["max_wells"]]

    n_val = max(1, int(len(files_train_all) * cfg["val_ratio"]))
    val_files = files_train_all[:n_val]
    train_files = files_train_all[n_val:]
    
    print(f"Loading {len(files_train_all)} wells total. (Original Train: {len(files_train)},")# Augmented: {len(files_aug)}).")
    
    def get_arrays(files):
        feats, tgts = [], []
        for filename in files:
            try:
                #aug = filename in files_aug
                df = preprocess(load_well(cfg["data_dir"], filename, split="train",))# augmented=aug))
                df = df.filter(pl.col(TARGET).is_not_null())
                if len(df) == 0: continue
                
                feats.append(df.select(FEATURE_COLS).to_numpy().astype(np.float64))
                tgts.append(df.select(TARGET).to_numpy().ravel().astype(np.float64))
            except Exception as e:
                print(f"  Skip {filename}: {e}")
        
        if not feats:
            return np.array([]), np.array([])
        return np.concatenate(feats), np.concatenate(tgts)

    X_train_raw, y_train_raw = get_arrays(train_files)
    X_val_raw, y_val_raw = get_arrays(val_files)
    
    scaler = {
        "feature_cols": FEATURE_COLS,
        "normalized": False
    }

    # Cast to float32 ONLY for model input
    return np.array(X_train_raw, dtype=np.float32), np.array(y_train_raw, dtype=np.float32), \
           np.array(X_val_raw, dtype=np.float32), np.array(y_val_raw, dtype=np.float32), \
           y_val_raw, scaler

print("Preparing data (RAW SCALE)...")
X_train, y_train, X_val, y_val, y_val_raw, scaler = prepare_data(CONFIG)
print(f"Train size: {X_train.shape}, Val size: {X_val.shape}")

with open(CONFIG["scaler_path"], "wb") as f:
    pickle.dump(scaler, f)
print(f"Metadata saved to {CONFIG['scaler_path']}")


Preparing data (RAW SCALE)...
Loading 773 wells total. (Original Train: 773,
Train size: (4054898, 6), Val size: (1037357, 6)
Metadata saved to /home/samer/Documents/competitions/ROGII/outputs/dense_scaler_params.pkl


### 5. Build & Train Dense Model
Construct the MLP architecture and execute training with callbacks.

In [ ]:
# Cell 5: Build & train Dense model

def build_dense_model(input_shape, cfg):
    """Builds a simple Multi-Layer Perceptron (MLP)."""
    inp = keras.Input(shape=input_shape)
    x = inp
    for units in cfg["hidden_layers"]:
        x = layers.Dense(units, activation="leaky_relu", kernel_regularizer=regularizers.L2(cfg["kr_rate"]))(x)
        if cfg["dropout"] > 0:
            x = layers.Dropout(cfg["dropout"])(x)
    
    out = layers.Dense(1, activation="linear")(x)
    m = keras.Model(inp, out, name="Dense_TVT")
    
    m.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate= cfg["lr"], 
            global_clipnorm=cfg["gcn"],
            beta_1=cfg["beta_1"],
            beta_2=cfg["beta_2"],
            amsgrad=cfg["amsgrad"],
            weight_decay=cfg["weight_decay"],
            use_ema=True,
            ema_momentum=cfg["ema_momentum"],
            ),
        loss="huber",
        metrics=[keras.metrics.RootMeanSquaredError(name="rmse")],
    )
    return m

model = build_dense_model((len(FEATURE_COLS),), CONFIG)
model.summary()

cbs = [
    callbacks.ModelCheckpoint(CONFIG["model_path"], monitor="val_rmse", save_best_only=True, mode="min"),
    callbacks.ReduceLROnPlateau(monitor="val_rmse", factor=0.5, patience=3, min_lr=1e-6),
    # callbacks.EarlyStopping(monitor="val_rmse", patience=7, restore_best_weights=True)
]

print(f"\nStarting training...")
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=CONFIG["epochs"],
    batch_size=CONFIG["batch_size"],
    callbacks=cbs
)


Model: "Dense_TVT"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 6)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,985 (39.00 KB)

 Trainable params: 9,601 (37.50 KB)

 Non-trainable params: 384 (1.50 KB)


Starting training...
Epoch 1/100
31679/31679 ━━━━━━━━━━━━━━━━━━━━ 62s 2ms/step - loss: 11492.0332 - rmse: 11510.1914 - val_loss: 11546.0947 - val_rmse: 11564.8174 - learning_rate: 5.0000e-07
Epoch 2/100
31679/31679 ━━━━━━━━━━━━━━━━━━━━ 52s 2ms/step - loss: 11491.9287 - rmse: 11510.0166 - val_loss: 11545.9150 - val_rmse: 11564.6416 - learning_rate: 5.0000e-07
Epoch 3/100
19284/31679 ━━━━━━━━━━━━━━━━━━━━ 17s 1ms/step - loss: 11491.4818 - rmse: 11509.5708

### 6. Evaluate Model
Reload the best weights and calculate final validation RMSE on raw scale.

In [ ]:
# Cell 6: Evaluate (RAW SCALE)

best_model = keras.saving.load_model(CONFIG["model_path"])
yp = best_model.predict(X_val, batch_size=1024).ravel()

err = yp - y_val_raw
rmse = float(np.sqrt(np.mean(err ** 2)))
print(f"\nFinal Val RMSE (Dense model, RAW SCALE): {rmse:.4f}")


1926/1926 ━━━━━━━━━━━━━━━━━━━━ 2s 857us/step


KeyError: 'target_std'

### 6. Evaluate Model
Reload the best weights and calculate final validation RMSE on raw scale.

In [ ]:
# Cell 6: Evaluate (RAW SCALE)

best_model = keras.saving.load_model(CONFIG["model_path"])
yp = best_model.predict(X_val, batch_size=1024).ravel()

err = yp - y_val_raw
rmse = float(np.sqrt(np.mean(err ** 2)))
print(f"\nFinal Val RMSE (Dense model, RAW SCALE): {rmse:.4f}")


1926/1926 ━━━━━━━━━━━━━━━━━━━━ 2s 857us/step

Final Val RMSE (Dense model, RAW SCALE): 113.7148
